In [1]:
import logging
import ndjson
from pickhardtpayments.pickhardtpayments import *
from pickhardtpayments.pickhardtpayments.Payment import Payment
from pickhardtpayments.pickhardtpayments.Payment import logger

In [2]:
# *** Setup ***
# graph = ChannelGraph("./pickhardtpayments/channels.sample.json")
graph = ChannelGraph("./pickhardtpayments/listchannels20220412.json")
payment_amount = 100000

def only_channels_with_return_channels(_graph: ChannelGraph):
    channels_with_no_return_channel = []
    for edge in _graph.network.edges:
        if not _graph.network.has_edge(edge[1], edge[0]):
            channels_with_no_return_channel.append(edge)

    for edge in channels_with_no_return_channel:
        _graph.network.remove_edge(edge[0], edge[1], edge[2])

    if len(channels_with_no_return_channel) == 0:
        logging.debug("channel graph only had channels in both directions.")
    else:
        logging.debug("channel graph had unannounced channels.")
    return _graph

graph = only_channels_with_return_channels(graph)
uncertainty_network = UncertaintyNetwork(graph)
oracle_lightning_network = OracleLightningNetwork(graph)

In [3]:
sim_session = SyncSimulatedPaymentSession(oracle_lightning_network, uncertainty_network, prune_network=False)
print("Setup finished")

Setup finished


In [4]:
import random
def create_payment_set(_uncertainty_network, _number_of_payments, payment_amount) -> list[dict]:
    if (len(_uncertainty_network.network.nodes())) < 3:
        logging.warning("graph has less than two nodes")
        exit(-1)
    _payments = []
    while len(_payments) < _number_of_payments:
        # casting _channel_graph to list to avoid deprecation warning for python 3.9
        _random_nodes = random.sample(list(_uncertainty_network.network.nodes), 2)
        # additional check for existence; doing it here avoids the check in each round, improving runtime
        src_exists = _random_nodes[0] in _uncertainty_network.network.nodes()
        dest_exists = _random_nodes[1] in _uncertainty_network.network.nodes()
        if src_exists and dest_exists:
            p = {"sender": _random_nodes[0], "receiver": _random_nodes[1], "amount": amount}
            _payments.append(p)
    # write payments to file
    ndjson.dump(_payments, open("data/payments.ndjson", "w"))
    return 0


# + Creation of a collection of N payments (src, rcv, amount)
# create_payment_set(uncertainty_network, 200, payment_amount3)

# testing how many payments can be sent via dijkstra pay

In [5]:
# set level of verbosity
loglevel = "error"
numeric_level = logging.getLevelName(loglevel.upper())
if not isinstance(numeric_level, int):
    raise ValueError('Invalid log level: %s' % loglevel)
#logger.setLevel(60)


In [17]:
# start simulation
c = 0
successful_payments = 0
failed_payments = 0
payment_simulation = []

payment_set = ndjson.load(open("data/payments.ndjson", "r"))

for payment in payment_set:
    c += 1
    # create new payment session
    sim_session = SyncSimulatedPaymentSession(oracle_lightning_network, uncertainty_network, prune_network=False)
    # we need to make sure we forget all learnt information on the Uncertainty Network
    sim_session.forget_information()
    # logger.info("*********** Payment {} ***********".format(c))
    # logger.debug(f"now sending {payment['amount']} sats from {payment['sender']} to {payment['receiver']}")

    ret = sim_session.pay(payment["sender"], payment["receiver"], 1000, loglevel=loglevel)
    if ret > 0:
        successful_payments += 1
        logger.error("Payment in run {} was successful.".format(c))
    if ret < 0:
        failed_payments += 1
        logger.error("Payment in run {} failed.".format(c))
    payment['success'] = ret
    payment_simulation.append(payment)

ndjson.dump(payment_simulation, open("data/payment_sim.ndjson", "w"))
print(f"\n{c} payments. {successful_payments} successful, {failed_payments} failed.")


KeyboardInterrupt: 

# testing how many payments can be sent via pickhardt pay

In [10]:
# start simulation
c = 0
successful_payments = 0
failed_payments = 0
payment_simulation = []

payment_set = ndjson.load(open("data/payments.ndjson", "r"))

for payment in payment_set:
    c += 1
    # create new payment session
    sim_session = SyncSimulatedPaymentSession(oracle_lightning_network, uncertainty_network, prune_network=False)
    # we need to make sure we forget all learnt information on the Uncertainty Network
    sim_session.forget_information()
    # logger.info("*********** Payment {} ***********".format(c))
    # logger.debug(f"now sending {payment['amount']} sats from {payment['sender']} to {payment['receiver']}")

    ret = sim_session.pay(payment["sender"], payment["receiver"], 1000,
                                    mu=0, base=0, loglevel=loglevel)
    if ret > 0:
        successful_payments += 1
        logger.error("Payment in run {} was successful.".format(c))
    if ret < 0:
        failed_payments += 1
        logger.error("Payment in run {} failed.".format(c))
    payment['success'] = ret
    payment_simulation.append(payment)

ndjson.dump(payment_simulation, open("data/payment_sim.ndjson", "w"))
print(f"\n{c} payments. {successful_payments} successful, {failed_payments} failed.")

# 19/200 successful with amount 1.000.000
# 25/200 successful with amount 100.000
# 89/200 successful with amount 1.000


200 payments. 0 successful, 0 failed.


In [8]:
print("===PICKHARDT PAY===")
sim_session.forget_information()
p = Payment(uncertainty_network, oracle_lightning_network, _random_nodes[0], _random_nodes[1], 30000, mu=0, base=0)
sim_session.pickhardt_pay(p.sender, p.receiver, p.total_amount, mu=0, base=0)
print("=====")


===PICKHARDT PAY===


NameError: name '_random_nodes' is not defined

In [ ]:
import networkx as nx

degree_centrality = nx.algorithms.centrality.degree_centrality(uncertainty_network.network)
for i, w in enumerate(sorted(degree_centrality, key=degree_centrality.get, reverse=True)):
    if(i<10):
        print(w, degree_centrality[w])
    else:
        break

In [ ]:
#------------------------------

In [ ]:
from ortools.graph import pywrapgraph
#print(nx.get_node_attributes(G,'pos'))
#G.dijkstra = pywrapgraph.DijkstraShortestPath(G.number_of_nodes(), 'A', 'B', G, 5000000)

In [ ]:
G = nx.MultiDiGraph()
for edge in uncertainty_network.network.edges(data="channel", keys=True):
    if edge[3].capacity > p._total_amount:
        G.add_edge(edge[0], edge[1], weight=edge[3].ppm)
        #print("edge added with cost ", edge[3].ppm)
    # if edge[3].capacity <= self._total_amount:
    #     print("eliminated edge: ", edge)

In [ ]:
path = nx.dijkstra_path(G, _random_nodes[0], _random_nodes[1])
print("shortest path: ", path)